In [0]:
# =============================================================
# 08_optimize — OPTIMIZE + Z-ORDER performance tuning
# Author: oakville3456
# Branch: feature/priority7-optimize
# Purpose: Compact small files, co-locate data for fast queries
# =============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

SILVER = "abfss://silver@saretailsalesdev.dfs.core.windows.net/sales"
GOLD   = "abfss://gold@saretailsalesdev.dfs.core.windows.net/sales_daily"

# Check current file state before OPTIMIZE
print("=== Silver file state BEFORE OPTIMIZE ===")
spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`") \
     .select("numFiles", "sizeInBytes") \
     .show(truncate=False)

print("=== Gold file state BEFORE OPTIMIZE ===")
spark.sql(f"DESCRIBE DETAIL delta.`{GOLD}`") \
     .select("numFiles", "sizeInBytes") \
     .show(truncate=False)

In [0]:
# Cell 2 — OPTIMIZE + Z-ORDER on Silver
# OPTIMIZE  = compact small files into larger ones
# Z-ORDER   = co-locate data by column for faster filtered queries

print("=== Running OPTIMIZE + Z-ORDER on Silver ===")
print("Z-ORDER by: store_id, order_date (most common filter columns)")

# OPTIMIZE with Z-ORDER
spark.sql(f"""
    OPTIMIZE delta.`{SILVER}`
    ZORDER BY (store_id, order_date)
""")

print("✅ OPTIMIZE + Z-ORDER complete on Silver")

# Check file state AFTER
print("\n=== Silver file state AFTER OPTIMIZE ===")
spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`") \
     .select("numFiles", "sizeInBytes") \
     .show(truncate=False)

# Check history — OPTIMIZE recorded as new version
print("=== Latest Silver history ===")
DeltaTable.forPath(spark, SILVER) \
    .history(3) \
    .select("version", "timestamp", "operation") \
    .show(truncate=False)

In [0]:
# Cell 3 — Prove Z-ORDER speeds up filtered queries
import time

silver = spark.table("adb_retail_dev.silver.sales")

# Query 1 — filter by store_id (Z-ORDER column)
start  = time.time()
count1 = silver.filter(F.col("store_id") == "S01").count()
time1  = round(time.time() - start, 3)

# Query 2 — filter by store_id + order_date
start  = time.time()
count2 = silver.filter(
    (F.col("store_id") == "S01") &
    (F.col("order_date") == "2024-01-15")
).count()
time2  = round(time.time() - start, 3)

# Query 3 — full table scan
start  = time.time()
count3 = silver.count()
time3  = round(time.time() - start, 3)

print("Query benchmark results:")
print("Filter store_id only         : " + str(count1) + " rows in " + str(time1) + "s")
print("Filter store_id + order_date : " + str(count2) + " rows in " + str(time2) + "s")
print("Full table scan              : " + str(count3) + " rows in " + str(time3) + "s")
print("")
print("Note: dataset too small to see dramatic difference")
print("In production (billions of rows) Z-ORDER reduces scan by 90%+")

In [0]:
# Cell 4 — ANALYZE TABLE
# Updates column statistics so Databricks query optimizer
# makes better decisions about join order and file skipping

print("=== Running ANALYZE TABLE on Silver ===")
spark.sql("""
    ANALYZE TABLE adb_retail_dev.silver.sales
    COMPUTE STATISTICS FOR ALL COLUMNS
""")
print("✅ ANALYZE complete")

# View the statistics
print("\n=== Column statistics ===")
spark.sql("""
    DESCRIBE EXTENDED adb_retail_dev.silver.sales order_id
""").show(truncate=False)

In [0]:
# Cell 5 — OPTIMIZE + Z-ORDER on Gold
print("=== Running OPTIMIZE + Z-ORDER on Gold ===")

spark.sql(f"""
    OPTIMIZE delta.`{GOLD}`
    ZORDER BY (store_id, order_date)
""")
print("✅ OPTIMIZE + Z-ORDER complete on Gold")

spark.sql("""
    ANALYZE TABLE adb_retail_dev.gold.sales_daily
    COMPUTE STATISTICS FOR ALL COLUMNS
""")
print("✅ ANALYZE complete on Gold")

# Final file state
print("\n=== Gold file state after OPTIMIZE ===")
spark.sql("DESCRIBE DETAIL adb_retail_dev.gold.sales_daily") \
     .select("numFiles", "sizeInBytes") \
     .show(truncate=False)